In [1]:
# ============================================================
# RAG IMPLEMENTATION — OPENAI NATIVE 
# ============================================================
# Documents: NIST AI RMF PDF + Reid Blackman podcast transcript
# ============================================================

# ── STEP 1: SETUP ─────────────────────────────────────────────────

import os
import json
import time
import numpy as np
from typing import List, Dict, Tuple
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Verify
api_key = os.getenv("OPENAI_API_KEY")
print("=" * 55)
print("STEP 1: SETUP")
print("=" * 55)
print(f"API key: {'✓ loaded' if api_key else '✗ missing'}")

STEP 1: SETUP
API key: ✓ loaded


In [3]:
# ── Load and chunk documents ──────────────────────────────────────

def recursive_chunk(text, chunk_size=1000, overlap=200,
                    separators=None):
    """Recursive character chunking — from previous lab."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", "! ", "? ", " ", ""]

    def split_recursive(text, separators):
        if len(text) <= chunk_size:
            return [text] if text.strip() else []
        for i, sep in enumerate(separators):
            if sep == "":
                return [text[j:j+chunk_size]
                        for j in range(0, len(text), chunk_size - overlap)]
            if sep in text:
                parts   = text.split(sep)
                chunks  = []
                current = ""
                for part in parts:
                    candidate = current + (sep if current else "") + part
                    if len(candidate) <= chunk_size:
                        current = candidate
                    else:
                        if current.strip():
                            chunks.append(current)
                        if len(part) > chunk_size:
                            chunks.extend(
                                split_recursive(part, separators[i+1:]))
                            current = ""
                        else:
                            current = part
                if current.strip():
                    chunks.append(current)
                return chunks if chunks else [text]
        return [text]

    return split_recursive(text, separators)


# Load documents
with open("trustworthy_ai_podcast.txt", "r", encoding="utf-8") as f:
    podcast_text = f.read()
with open("nist_ai_rmf.txt", "r", encoding="utf-8") as f:
    pdf_text = f.read()

# Chunk both documents
podcast_chunks = recursive_chunk(podcast_text, 800, 150)
pdf_chunks     = recursive_chunk(pdf_text,     800, 150)

# Build document store with metadata
documents = []
for i, chunk in enumerate(pdf_chunks):
    documents.append({
        "id":     f"pdf_{i}",
        "text":   chunk,
        "source": "NIST AI RMF",
        "type":   "pdf",
        "index":  i
    })
for i, chunk in enumerate(podcast_chunks):
    documents.append({
        "id":     f"pod_{i}",
        "text":   chunk,
        "source": "Reid Blackman — AI Ethics Podcast",
        "type":   "podcast",
        "index":  i
    })

print(f"\nSTEP 1: DOCUMENT PREPARATION")
print(f"  PDF chunks:     {len(pdf_chunks)}")
print(f"  Podcast chunks: {len(podcast_chunks)}")
print(f"  Total docs:     {len(documents)}")


STEP 1: DOCUMENT PREPARATION
  PDF chunks:     133
  Podcast chunks: 63
  Total docs:     196


In [4]:
# ── STEP 2: GENERATE EMBEDDINGS ───────────────────────────────────

def get_embeddings_batch(texts: List[str],
                          model: str = "text-embedding-3-small",
                          batch_size: int = 100) -> List[List[float]]:
    """Generate embeddings in batches for efficiency."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch    = texts[i:i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Embedded {min(i+batch_size, len(texts))}"
              f"/{len(texts)} chunks")
        if i + batch_size < len(texts):
            time.sleep(0.1)   # avoid rate limits
    return all_embeddings


EMBEDDING_MODEL = "text-embedding-3-small"
EMBED_CACHE     = "embeddings_cache.json"

# Use cache to avoid re-embedding on every run
if os.path.exists(EMBED_CACHE):
    print(f"\nSTEP 2: Loading embeddings from cache...")
    with open(EMBED_CACHE, "r") as f:
        cached = json.load(f)
    embeddings = cached["embeddings"]
    print(f"  ✓ Loaded {len(embeddings)} embeddings from cache")
else:
    print(f"\nSTEP 2: Generating embeddings ({len(documents)} chunks)...")
    print(f"  Model: {EMBEDDING_MODEL}")
    print(f"  Estimated cost: ~${len(documents)*0.00002:.4f}")

    texts      = [doc["text"] for doc in documents]
    embeddings = get_embeddings_batch(texts, EMBEDDING_MODEL)

    # Save to cache
    with open(EMBED_CACHE, "w") as f:
        json.dump({"embeddings": embeddings,
                   "model": EMBEDDING_MODEL}, f)
    print(f"  ✓ Generated and cached {len(embeddings)} embeddings")

# Attach embeddings to documents
for doc, emb in zip(documents, embeddings):
    doc["embedding"] = emb

# Convert to numpy array for fast similarity computation
embedding_matrix = np.array(embeddings, dtype=np.float32)
print(f"  Embedding matrix shape: {embedding_matrix.shape}")
print(f"  Embedding dimensions:   {embedding_matrix.shape[1]}")


STEP 2: Generating embeddings (196 chunks)...
  Model: text-embedding-3-small
  Estimated cost: ~$0.0039
  Embedded 100/196 chunks
  Embedded 196/196 chunks
  ✓ Generated and cached 196 embeddings
  Embedding matrix shape: (196, 1536)
  Embedding dimensions:   1536


In [5]:
# ── STEP 3: VECTOR SEARCH ─────────────────────────────────────────

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between vector a and matrix b."""
    a_norm = a / (np.linalg.norm(a) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return b_norm @ a_norm


def retrieve(query: str,
             top_k: int = 5,
             source_filter: str = None) -> List[Dict]:
    """
    Find the most relevant chunks for a query.
    
    Parameters:
    - query:         user question
    - top_k:         number of chunks to return
    - source_filter: 'pdf', 'podcast', or None for both
    
    Returns:
    - list of top_k most relevant document dicts with scores
    """
    # Embed the query
    response      = client.embeddings.create(
        model=EMBEDDING_MODEL, input=[query])
    query_vector  = np.array(response.data[0].embedding, dtype=np.float32)

    # Calculate similarities
    similarities  = cosine_similarity(query_vector, embedding_matrix)

    # Apply source filter
    if source_filter:
        mask         = np.array([d["type"] == source_filter
                                 for d in documents])
        similarities = np.where(mask, similarities, -1)

    # Get top_k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        doc = documents[idx].copy()
        doc["score"] = float(similarities[idx])
        doc.pop("embedding", None)   # remove for readability
        results.append(doc)

    return results


# Test retrieval
print("\nSTEP 3: VECTOR SEARCH TEST")
print("=" * 55)
test_query = "What are the key characteristics of trustworthy AI?"
results    = retrieve(test_query, top_k=3)
print(f"Query: '{test_query}'")
print(f"\nTop 3 results:")
for i, r in enumerate(results, 1):
    print(f"\n  [{i}] Score: {r['score']:.4f} | Source: {r['source']}")
    print(f"       {r['text'][:150].strip()}...")


STEP 3: VECTOR SEARCH TEST
Query: 'What are the key characteristics of trustworthy AI?'

Top 3 results:

  [1] Score: 0.7331 | Source: NIST AI RMF
       Trustworthiness characteristics explained in this document influence each other.
Highly secure but unfair systems, accurate but opaque and uninterpret...

  [2] Score: 0.7168 | Source: NIST AI RMF
       tributes, accountability and transparency also relate to the processes and activities internal
to an AI system and its external setting. Neglecting th...

  [3] Score: 0.7088 | Source: NIST AI RMF
       tion, and validation tasks. Note that AI actors in the AI Model dimension
(Figure2)areseparatedasabestpractice,withthosebuildingandusingthe
modelssepa...


In [6]:
# ── STEP 4: RAG QUERY FUNCTION ────────────────────────────────────

def rag_query(question: str,
              top_k: int = 5,
              model: str = "gpt-4o",
              source_filter: str = None,
              verbose: bool = True) -> Dict:
    """
    Complete RAG pipeline:
    1. Retrieve relevant chunks
    2. Format context
    3. Generate answer with citations
    
    Parameters:
    - question:      user question
    - top_k:         chunks to retrieve
    - model:         OpenAI chat model
    - source_filter: 'pdf', 'podcast', or None
    - verbose:       print retrieved chunks
    
    Returns:
    - dict with answer, sources, and chunks used
    """
    # Step 1 — Retrieve
    chunks = retrieve(question, top_k=top_k,
                      source_filter=source_filter)

    if verbose:
        print(f"\nRetrieved {len(chunks)} chunks:")
        for i, c in enumerate(chunks, 1):
            print(f"  [{i}] {c['source']} (score: {c['score']:.3f})")
            print(f"       {c['text'][:100].strip()}...")

    # Step 2 — Format context with source labels
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Source {i}: {chunk['source']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Step 3 — Generate answer
    system_prompt = """You are an expert assistant on Trustworthy AI.
Answer questions using ONLY the provided context.
Always cite your sources using [Source N] notation.
If the context does not contain enough information, say so clearly.
Be concise but thorough."""

    user_prompt = f"""Context:
{context}

Question: {question}

Answer with citations:"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=600
    )

    answer = response.choices[0].message.content

    return {
        "question":    question,
        "answer":      answer,
        "sources":     [{"source": c["source"],
                         "score":  round(c["score"], 4),
                         "text":   c["text"][:200]}
                        for c in chunks],
        "model":       model,
        "chunks_used": len(chunks),
        "tokens_used": response.usage.total_tokens
    }

In [7]:
# ── EXAMPLE QUERIES ───────────────────────────────────────────────

print("\n" + "=" * 60)
print("STEP 4: RAG QUERY EXAMPLES")
print("=" * 60)

queries = [
    {
        "q":      "What are the key characteristics of trustworthy AI?",
        "filter": None,
        "desc":   "Both sources"
    },
    {
        "q":      "How should organizations govern AI risk management?",
        "filter": "pdf",
        "desc":   "PDF only (NIST)"
    },
    {
        "q":      "What is the biggest misconception companies have "
                  "about AI ethics?",
        "filter": "podcast",
        "desc":   "Podcast only (Reid Blackman)"
    },
    {
        "q":      "How does bias appear in AI hiring systems "
                  "and how can it be fixed?",
        "filter": None,
        "desc":   "Both sources"
    },
]

all_results = []

for i, query in enumerate(queries, 1):
    print(f"\n{'─'*60}")
    print(f"QUERY {i} [{query['desc']}]")
    print(f"Q: {query['q']}")
    print(f"{'─'*60}")

    result = rag_query(
        question=query["q"],
        top_k=4,
        model="gpt-4o",
        source_filter=query["filter"],
        verbose=False   # set True to see retrieved chunks
    )

    print(f"\nA: {result['answer']}")
    print(f"\nSources used ({result['chunks_used']} chunks, "
          f"{result['tokens_used']} tokens):")
    for s in result["sources"][:3]:
        print(f"  • {s['source']} (similarity: {s['score']})")

    all_results.append(result)
    time.sleep(0.5)

print(f"\n{'='*60}")
print("✓ All queries complete")


STEP 4: RAG QUERY EXAMPLES

────────────────────────────────────────────────────────────
QUERY 1 [Both sources]
Q: What are the key characteristics of trustworthy AI?
────────────────────────────────────────────────────────────

A: The key characteristics of trustworthy AI, as outlined in the NIST AI RMF, include:

1. **Valid and Reliable**: This is considered a necessary condition for trustworthiness and serves as the foundation for other characteristics [Source 2].

2. **Safe, Secure, and Resilient**: Ensuring that AI systems are protected against threats and can withstand and recover from adverse conditions [Source 4].

3. **Accountable and Transparent**: These characteristics relate to both the internal processes of an AI system and its external interactions, and they are crucial for reducing the probability and magnitude of negative consequences [Source 2].

4. **Explainable and Interpretable**: AI systems should be understandable to humans, allowing users to comprehend how decis

In [8]:
# ── Save results ──────────────────────────────────────────────────

with open("rag_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print("✓ Saved rag_results.json")

# Print design choices summary
print("""
DESIGN CHOICES:
──────────────────────────────────────────────────────────
1. Embedding model: text-embedding-3-small
   - 1536 dimensions, fast, cheap ($0.00002/1K tokens)
   - Sufficient quality for this document size

2. Chunking: Recursive 800 chars / 150 overlap
   - Preserves sentence boundaries (from Step 7 analysis)
   - 150 char overlap ensures context continuity

3. Similarity: Cosine similarity (numpy)
   - No external vector DB needed
   - Fast enough for < 10,000 chunks

4. Retrieval: top_k=4 chunks per query
   - Enough context without exceeding token limits
   - Source filter allows targeting PDF or podcast

5. Generation: gpt-4o with temperature=0.1
   - Low temperature for factual accuracy
   - System prompt enforces citation format

6. Caching: embeddings saved to JSON
   - Avoids re-embedding on every run
   - Saves API costs during development
""")

✓ Saved rag_results.json

DESIGN CHOICES:
──────────────────────────────────────────────────────────
1. Embedding model: text-embedding-3-small
   - 1536 dimensions, fast, cheap ($0.00002/1K tokens)
   - Sufficient quality for this document size

2. Chunking: Recursive 800 chars / 150 overlap
   - Preserves sentence boundaries (from Step 7 analysis)
   - 150 char overlap ensures context continuity

3. Similarity: Cosine similarity (numpy)
   - No external vector DB needed
   - Fast enough for < 10,000 chunks

4. Retrieval: top_k=4 chunks per query
   - Enough context without exceeding token limits
   - Source filter allows targeting PDF or podcast

5. Generation: gpt-4o with temperature=0.1
   - Low temperature for factual accuracy
   - System prompt enforces citation format

6. Caching: embeddings saved to JSON
   - Avoids re-embedding on every run
   - Saves API costs during development



WHAT DO WE GET AS A RESULT?
A complete RAG system with no vector database required — embeddings stored in memory as a numpy matrix, cosine similarity computed directly. Includes caching so embeddings are only generated once, source filtering to query PDF or podcast independently, and automatic citations in the answer.

# ============================================================
# DESIGN CHOICES EXPLANATION
# ============================================================

design_choices = """
# RAG System Design Choices
## Lab: RAG with OpenAI Native APIs
## Documents: NIST AI RMF PDF + Reid Blackman Podcast Transcript

---

## 1. Chunking Strategy — Recursive 800 chars / 150 overlap

CHOICE: Recursive character chunking with 800 character chunks
and 150 character overlap.

WHY:
From the Step 7 quality analysis, recursive chunking achieved
near-zero mid-word breaks compared to 30-40% for fixed-size.
It respects natural boundaries — paragraph breaks, sentence
endings — by trying separators in priority order before
resorting to hard character splits.

800 characters was chosen as a balance between:
- Large enough to contain a complete thought or argument
- Small enough to keep retrieval precise (not too much noise)
- Approximately 200 tokens — well within embedding model limits

150 character overlap ensures that context spanning a chunk
boundary is not lost — a sentence that starts at the end of
chunk N will also appear at the start of chunk N+1.

ALTERNATIVE CONSIDERED: Semantic chunking gave better boundary
quality but is too slow for a lab environment. Recursive is
the best practical choice.

---

## 2. Embedding Model — text-embedding-3-small

CHOICE: OpenAI text-embedding-3-small (1536 dimensions)

WHY:
- Cost: $0.00002 per 1K tokens — approximately 5x cheaper
  than text-embedding-3-large
- Speed: faster generation, better for prototyping
- Quality: sufficient for a focused domain corpus like
  Trustworthy AI — the documents are topically coherent
  so the model does not need maximum discriminative power
- Dimensions: 1536 is the right balance between expressiveness
  and memory/compute cost for cosine similarity

WHEN TO UPGRADE: Use text-embedding-3-large for production
RAG over large, diverse document collections where subtle
semantic distinctions matter more.

---

## 3. Vector Store — Numpy cosine similarity (no Pinecone)

CHOICE: In-memory numpy matrix with cosine similarity search.

WHY:
- The corpus is small (~50-100 chunks total) so a vector
  database would add complexity with no performance benefit
- numpy cosine similarity over 100 vectors takes microseconds
- No external service dependency — no API key, no latency,
  no cost, no setup
- Embeddings cached to JSON so they are only generated once

WHEN TO USE PINECONE INSTEAD:
- Corpus exceeds ~50,000 chunks
- Multiple users querying simultaneously
- Need persistent storage across sessions
- Need metadata filtering at scale

---

## 4. Retrieval — top_k=4 with source filtering

CHOICE: Retrieve top 4 most similar chunks. Optional source
filter to query PDF or podcast independently.

WHY top_k=4:
- 4 chunks × ~800 chars ≈ ~3,200 chars ≈ ~800 tokens of context
- Leaves plenty of room in gpt-4o's 128K context window
- Enough context for nuanced answers without irrelevant noise
- Increasing top_k beyond 5-6 typically adds noise rather
  than useful context

WHY source filtering:
- The two documents serve different purposes:
  NIST PDF → authoritative, structured, regulatory guidance
  Podcast  → practical, conversational, real-world examples
- Allowing the user to target one source improves precision
  for specific question types

---

## 5. Generation — gpt-4o, temperature=0.1

CHOICE: gpt-4o with temperature=0.1 and mandatory citation
format in the system prompt.

WHY gpt-4o:
- Better instruction following than gpt-4o-mini
- More reliable at citing sources correctly
- Better at synthesising information across multiple chunks

WHY temperature=0.1:
- RAG is a factual retrieval task — we want deterministic,
  grounded answers, not creative variation
- Low temperature reduces hallucination risk
- The model should report what the context says, not
  invent additional information

WHY citation format:
- [Source N] notation makes answers verifiable
- Users can check which document supports each claim
- Essential for trustworthy AI applications — ironic but
  appropriate given the topic

---

## 6. Caching — embeddings_cache.json

CHOICE: Save embeddings to a local JSON file after first run.

WHY:
- Embedding 100 chunks costs ~$0.002 but takes 5-10 seconds
- During development you run the notebook many times
- Caching saves both time and money
- In production this would be a proper vector database

---

## Summary

| Component      | Choice                    | Key Reason                    |
|---------------|---------------------------|-------------------------------|
| Chunking      | Recursive 800/150         | Best boundary quality (Step 7)|
| Embedding     | text-embedding-3-small    | Cost/quality balance          |
| Vector store  | Numpy in-memory           | Corpus too small for Pinecone |
| top_k         | 4 chunks                  | Context/noise balance         |
| Generation    | gpt-4o temp=0.1           | Factual accuracy + citations  |
| Caching       | JSON file                 | Save cost during development  |
"""

print(design_choices)

# Save as markdown file for submission
with open("design_choices.md", "w", encoding="utf-8") as f:
    f.write(design_choices)
print("✓ Saved design_choices.md")